In [1]:
from entail_tool import *
import pickle as pkl
import pandas as pd
import numpy as np
import numpy as np
from tqdm import tqdm
import argparse

def get_sentence_splitter():
    from sentence_splitter import SentenceSplitter
    _splitter = SentenceSplitter(language='en')
    return _splitter

In [2]:
import gc
import torch

def list_gpu_tensors_by_creation_order():
    tensors = []
    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj) and obj.is_cuda:
                tensors.append(obj)
        except:
            pass
    # Sort tensors by their memory address (id), assuming higher id implies more recent creation
    tensors.sort(key=lambda x: id(x), reverse=True)
    for tensor in tensors[:10]:
        print(f"Type: {type(tensor)}, Size: {tensor.size()}, Memory: {tensor.element_size() * tensor.nelement() / 1024 ** 2:.2f} MB, ID: {id(tensor)}")

In [3]:
f = open(f'../../rag_utility/doc_dicts/nq_wiki_dict.pkl', 'rb')
doc_dict = pkl.load(f)
f.close()

In [4]:
res = pd.read_csv(f'../../rag_utility/res/mt5_nq_test.csv')

In [5]:
judger = EntailmentDeberta()

In [6]:
# judger.check_implication(['The weather is good', 'The weather is good and I like you'], ['The weather is good and I like you', 'The weather is good'])

In [7]:
_k = 3

splitter = get_sentence_splitter()

doc_texts = res[(res.qid == 'test_0') & (res['rank'] <_k)].docno.apply(lambda x: doc_dict[str(x)]).values
sentences = []
for doc_text in doc_texts:
    sentences += splitter.split(doc_text)
    

In [8]:
sentences[:2]

['the winner was an American student.',
 'In 2009, the prize went to Mor Tzaban, a high school student from Netivot, Israel.']

In [9]:
[sentences[0], sentences[1]]

['the winner was an American student.',
 'In 2009, the prize went to Mor Tzaban, a high school student from Netivot, Israel.']

In [10]:
type(sentences[0])

str

In [11]:
# judger.check_implication(2*[sentences[1]], 2*[sentences[2]])

In [12]:
len(sentences)

13

In [14]:
import timeit

s = timeit.default_timer()

for i in range(len(sentences)):

    print(i)
    judger.check_implication(len(sentences)*[sentences[i]], sentences)
    
e = timeit.default_timer()
    
print(e-s)

0
tensor([[2.9074e-04, 3.0024e-03, 9.9671e-01],
        [9.5068e-01, 4.9004e-02, 3.2101e-04],
        [8.6934e-01, 1.2976e-01, 9.0064e-04],
        [1.4143e-02, 9.8499e-01, 8.6502e-04],
        [1.3425e-03, 9.8922e-01, 9.4403e-03],
        [2.0444e-03, 9.8458e-01, 1.3378e-02],
        [3.5469e-03, 9.9288e-01, 3.5718e-03],
        [6.2102e-01, 3.7719e-01, 1.7931e-03],
        [5.2713e-02, 9.4257e-01, 4.7198e-03],
        [7.2610e-01, 2.7223e-01, 1.6749e-03],
        [1.2702e-03, 9.9734e-01, 1.3890e-03],
        [2.5889e-03, 9.9624e-01, 1.1680e-03],
        [2.0187e-02, 9.7793e-01, 1.8878e-03]], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)
1
tensor([[9.9911e-01, 5.9102e-04, 2.9501e-04],
        [3.2208e-04, 1.7497e-03, 9.9793e-01],
        [2.2150e-02, 9.7634e-01, 1.5136e-03],
        [2.5084e-03, 9.9715e-01, 3.4361e-04],
        [3.3432e-03, 9.2721e-01, 6.9451e-02],
        [2.8167e-03, 9.6452e-01, 3.2667e-02],
        [1.7637e-03, 9.9568e-01, 2.5600e-03],
        [2.3649e-01, 7.

In [13]:
# import gc
# import torch

# def list_gpu_tensors():
#     for obj in gc.get_objects():
#         try:
#             if torch.is_tensor(obj) and obj.is_cuda:
#                 print(f"Tensor: {type(obj)}, Size: {obj.size()}, Memory: {obj.element_size() * obj.nelement() / 1024 ** 2:.2f} MB")
#         except:
#             pass

# list_gpu_tensors()

In [14]:
# import gc

# gc.get_objects()

In [17]:
list_gpu_tensors_by_creation_order()

Type: <class 'torch.nn.parameter.Parameter'>, Size: torch.Size([1536, 6144]), Memory: 36.00 MB, ID: 139642328312112
Type: <class 'torch.Tensor'>, Size: torch.Size([2, 24, 15, 64]), Memory: 0.18 MB, ID: 139642292616944
Type: <class 'torch.Tensor'>, Size: torch.Size([2, 24, 15, 15]), Memory: 0.04 MB, ID: 139642292616656
Type: <class 'torch.Tensor'>, Size: torch.Size([48, 15, 64]), Memory: 0.18 MB, ID: 139642292616368
Type: <class 'torch.Tensor'>, Size: torch.Size([48, 15, 64]), Memory: 0.18 MB, ID: 139642292616176
Type: <class 'torch.Tensor'>, Size: torch.Size([2, 24, 15, 64]), Memory: 0.18 MB, ID: 139642292615984
Type: <class 'torch.nn.parameter.Parameter'>, Size: torch.Size([3]), Memory: 0.00 MB, ID: 139642292615888
Type: <class 'torch.nn.parameter.Parameter'>, Size: torch.Size([3, 1536]), Memory: 0.02 MB, ID: 139642292615792
Type: <class 'torch.nn.parameter.Parameter'>, Size: torch.Size([1536]), Memory: 0.01 MB, ID: 139642292615696
Type: <class 'torch.nn.parameter.Parameter'>, Size: t

/opt/miniconda3/envs/semantic_uncertainty_export/lib/python3.11/site-packages/torch/__init__.py:1113: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)


In [18]:

try:
    del x
    del y
    torch.cuda.empty_cache()
except:
    torch.cuda.empty_cache()

In [22]:
for i in range(len(sentences)):
    print(i)
    judger.check_implication([sentences[i]]*len(sentences), sentences)

In [24]:
list_gpu_tensors_by_creation_order()

Type: <class 'torch.Tensor'>, Size: torch.Size([13, 24, 58, 64]), Memory: 4.42 MB, ID: 139655557195600
Type: <class 'torch.Tensor'>, Size: torch.Size([13, 24, 58, 64]), Memory: 4.42 MB, ID: 139655557195024
Type: <class 'torch.Tensor'>, Size: torch.Size([13, 58]), Memory: 0.01 MB, ID: 139650827187984
Type: <class 'torch.Tensor'>, Size: torch.Size([512, 1536]), Memory: 3.00 MB, ID: 139650827187792
Type: <class 'torch.Tensor'>, Size: torch.Size([13, 1536]), Memory: 0.08 MB, ID: 139650827187600
Type: <class 'torch.Tensor'>, Size: torch.Size([13, 58, 1536]), Memory: 4.42 MB, ID: 139650827187408
Type: <class 'torch.nn.parameter.Parameter'>, Size: torch.Size([1536, 6144]), Memory: 36.00 MB, ID: 139642328312112
Type: <class 'torch.Tensor'>, Size: torch.Size([2, 24, 15, 64]), Memory: 0.18 MB, ID: 139642292616944
Type: <class 'torch.Tensor'>, Size: torch.Size([2, 24, 15, 15]), Memory: 0.04 MB, ID: 139642292616656
Type: <class 'torch.Tensor'>, Size: torch.Size([48, 15, 64]), Memory: 0.18 MB, ID: 